# Computer Vision — Lab 08: Object Detection Using Regional Convolutional Neural Networks (R-CNN)

**Department of Artificial Intelligence**  
**National University of Computer & Emerging Sciences (NUCES FAST)**  
**Course:** Computer Vision (CS / AI)  
**Task:** 6.1 Object Detection Using Regional Convolutional Neural Networks (R-CNN)  
**Dataset:** Raccoon Benchmark Object Detection Dataset (Pascal VOC XML Format)  
*Note: As instructed in Lab Task 6.1 ("Do not use the mentioned BCCD dataset; use another dataset of your choice"), this implementation employs the widely recognized Raccoon Object Detection benchmark dataset.*

---

## 1. Executive Summary & Theoretical Foundations

Object detection entails identifying **what** objects are present in an image and **where** they are located via bounding boxes.

Classic **R-CNN** (Regions with CNN features, Girshick et al., 2014) revolutionized modern computer vision by breaking down object detection into a modular, multi-stage paradigm:
1. **Region Proposal Generation:** Extracting prospective candidate object boundaries (~2,000 regions per image) using unsupervised hierarchical grouping (**Selective Search**).
2. **Feature Extraction:** Warping candidate regions into fixed dimensions ($128 \times 128$ or $224 \times 224$) and feeding them through a deep Convolutional Neural Network (**ResNet-18 / MobileNetV2 / Custom CNN**).
3. **Category Classification & Bounding Box Regression:**
   - **Classification Head:** Categorizes each proposal into Foreground Object or Background.
   - **Bounding Box Regressor:** Applies linear/smooth regression to refine candidate bounding box coordinates closer to the true object boundary.
4. **Post-Processing via Non-Maximum Suppression (NMS):** Eliminates redundant, overlapping candidate boxes around the same object, preserving only the highest confidence detection.

---

### Mathematical Formulations

#### 1. Intersection over Union (IoU)
$$\text{IoU}(A, B) = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)}$$

#### 2. Bounding Box Parameterization (Girshick et al., 2014)
Given a proposal $P = (P_x, P_y, P_w, P_h)$ and ground truth $G = (G_x, G_y, G_w, G_h)$:
$$t_x = \frac{G_x - P_x}{P_w}, \quad t_y = \frac{G_y - P_y}{P_h}$$
$$t_w = \log\left(\frac{G_w}{P_w}\right), \quad t_h = \log\left(\frac{G_h}{P_h}\right)$$

Inverse decoding at inference:
$$\hat{G}_x = P_w \hat{t}_x + P_x, \quad \hat{G}_y = P_h \hat{t}_y + P_y$$
$$\hat{G}_w = P_w \exp(\hat{t}_w), \quad \hat{G}_h = P_h \exp(\hat{t}_h)$$

#### 3. Multi-Task Joint Loss
$$\mathcal{L}_{total} = \mathcal{L}_{cls}(p, u) + \lambda [u \ge 1] \mathcal{L}_{reg}(t, v)$$
where $\mathcal{L}_{cls}$ is Cross-Entropy loss, $\mathcal{L}_{reg}$ is Smooth $L_1$ loss, and $[u \ge 1]$ is the indicator activating regression only on foreground objects.


In [ ]:
# 1. Imports and Environment Setup
import os
import sys
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"OpenCV Version:  {cv2.__version__}")
print(f"Execution Device: {device}")


In [ ]:
# 2. Dataset Loading and Ground Truth Inspection
from dataset import download_and_extract_raccoon_dataset, parse_voc_xml

dataset_dir = download_and_extract_raccoon_dataset()
img_dir = os.path.join(dataset_dir, "images")
ann_dir = os.path.join(dataset_dir, "annotations")

sample_idx = "raccoon-1"
xml_path = os.path.join(ann_dir, f"{sample_idx}.xml")
img_path = os.path.join(img_dir, f"{sample_idx}.jpg")

annotation = parse_voc_xml(xml_path)
sample_img = cv2.imread(img_path)
sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

print("Parsed Annotation:")
print(f"  Filename: {annotation['filename']}")
print(f"  Dimensions: {annotation['width']} x {annotation['height']}")
print(f"  Ground Truth Boxes: {annotation['boxes']}")

# Plot Image with Ground Truth Bounding Box
plt.figure(figsize=(8, 6))
vis_gt = sample_rgb.copy()
for box in annotation['boxes']:
    x1, y1, x2, y2 = box
    cv2.rectangle(vis_gt, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(vis_gt, "GT: Raccoon", (x1, max(25, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

plt.imshow(vis_gt)
plt.title("Sample Image with Ground Truth Annotation (Pascal VOC Format)", fontsize=13, fontweight='bold')
plt.axis("off")
plt.show()


In [ ]:
# 3. Selective Search Region Proposal Generation
from selective_search import generate_proposals, compute_iou

t0 = time.time()
proposals = generate_proposals(sample_img, mode='fast', max_proposals=200, scale=0.6)
elapsed = time.time() - t0

print(f"Generated {len(proposals)} candidate region proposals in {elapsed:.2f} seconds.")

# Visualize First 25 Proposals
vis_props = sample_rgb.copy()
for i, (x1, y1, x2, y2) in enumerate(proposals[:25]):
    color = tuple(np.random.randint(50, 255, size=3).tolist())
    cv2.rectangle(vis_props, (x1, y1), (x2, y2), color, 2)

plt.figure(figsize=(8, 6))
plt.imshow(vis_props)
plt.title(f"Selective Search: Top 25 Candidate Proposals (out of {len(proposals)})", fontsize=13, fontweight='bold')
plt.axis("off")
plt.show()


In [ ]:
# 4. IoU Analysis & Proposal Categorization
gt_box = annotation['boxes'][0]

ious = [compute_iou(p, gt_box) for p in proposals]
pos_proposals = [p for p, iou in zip(proposals, ious) if iou >= 0.5]
neg_proposals = [p for p, iou in zip(proposals, ious) if iou < 0.15]

print(f"Maximum IoU achieved by Selective Search: {max(ious):.4f}")
print(f"Positive Proposals (IoU >= 0.50): {len(pos_proposals)}")
print(f"Negative Background Proposals (IoU < 0.15): {len(neg_proposals)}")

# Plot IoU Distribution
plt.figure(figsize=(7, 4))
plt.hist(ious, bins=20, color='skyblue', edgecolor='black')
plt.axvline(0.5, color='green', linestyle='--', linewidth=2, label='Pos Threshold (0.5)')
plt.axvline(0.15, color='red', linestyle='--', linewidth=2, label='Neg Threshold (0.15)')
plt.title("Distribution of Proposal IoUs with Ground Truth", fontsize=12, fontweight='bold')
plt.xlabel("Intersection over Union (IoU)")
plt.ylabel("Proposal Count")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# 5. R-CNN Model Architecture & Multi-Task Loss Definition
from model import RCNNModel, RCNNLoss

model = RCNNModel(num_classes=2, backbone='resnet18', pretrained=True).to(device)
print(f"Initialized R-CNN with Backbone: {model.backbone_name}")
print(f"Feature Dimension: {model.in_features}")
print(f"Classification Head: {model.classifier}")
print(f"Bounding Box Regressor: {model.bbox_regressor}")

criterion = RCNNLoss(reg_weight=1.0)


In [ ]:
# 6. Load Dataset and Prepared Proposal Split
from dataset import prepare_dataset_splits, RCNNCropDataset, get_rcnn_transforms

train_samples, val_samples = prepare_dataset_splits(
    dataset_dir=dataset_dir,
    cache_file="data/rcnn_proposals_cache.pt",
    train_ratio=0.8,
    max_images=60
)

train_tf = get_rcnn_transforms(crop_size=128, is_train=True)
val_tf = get_rcnn_transforms(crop_size=128, is_train=False)

train_loader = DataLoader(RCNNCropDataset(train_samples, train_tf), batch_size=32, shuffle=True)
val_loader = DataLoader(RCNNCropDataset(val_samples, val_tf), batch_size=32, shuffle=False)

print(f"Training Proposals: {len(train_samples)} | Validation Proposals: {len(val_samples)}")


In [ ]:
# 7. Model Checkpoint Loading & Training Verification
checkpoint_path = "outputs/best_rcnn_model.pth"

if os.path.exists(checkpoint_path):
    print(f"Loading trained weights from {checkpoint_path}...")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded successfully! Best Validation Loss was: {ckpt['val_loss']:.4f}")
else:
    print("Checkpoint not found. Running training via train.py...")
    from train import train_rcnn
    class QuickArgs:
        epochs = 6
        batch_size = 32
        lr = 1e-4
        weight_decay = 1e-4
        backbone = 'resnet18'
        crop_size = 128
        reg_weight = 1.0
        max_images = 60
        output_dir = 'outputs'
        cache_file = 'data/rcnn_proposals_cache.pt'
        force_recompute = False
    model, history = train_rcnn(QuickArgs())


In [ ]:
# 8. Visualizing Training and Validation Loss Curves
loss_curve_img = "outputs/training_loss_curve.png"
if os.path.exists(loss_curve_img):
    plt.figure(figsize=(16, 5))
    img_curve = cv2.cvtColor(cv2.imread(loss_curve_img), cv2.COLOR_BGR2RGB)
    plt.imshow(img_curve)
    plt.axis("off")
    plt.show()


In [ ]:
# 9. Proposal Classification Confusion Matrix & Metrics
from evaluate import evaluate_proposals

results = evaluate_proposals(model, val_samples, crop_size=128, device=device, output_dir="outputs")
cm_img = "outputs/confusion_matrix.png"
if os.path.exists(cm_img):
    plt.figure(figsize=(6, 5))
    plt.imshow(cv2.cvtColor(cv2.imread(cm_img), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


In [ ]:
# 10. Quantitative Evaluation: Precision-Recall Curve & mAP @ 0.50
pr_curve_img = "outputs/precision_recall_curve.png"
if os.path.exists(pr_curve_img):
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(cv2.imread(pr_curve_img), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


In [ ]:
# 11. Visual Object Detection with Bounding Box Regression & NMS
from inference import run_inference

test_image = os.path.join(dataset_dir, "images", "raccoon-1.jpg")
test_xml = os.path.join(dataset_dir, "annotations", "raccoon-1.xml")
out_img = "outputs/detections/notebook_det_1.png"

detections, vis = run_inference(
    image_path=test_image,
    checkpoint_path=checkpoint_path,
    output_path=out_img,
    gt_xml_path=test_xml,
    score_thresh=0.60,
    nms_thresh=0.3
)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"R-CNN Detections with Regression Refinement & NMS (Detections: {len(detections)})", fontsize=13, fontweight='bold')
plt.axis("off")
plt.show()


## 12. Technical Discussion & Architectural Evolution

| Metric / Attribute | R-CNN (Girshick et al. 2014) | Fast R-CNN (Girshick 2015) | Faster R-CNN (Ren et al. 2015) | YOLO (Redmon et al. 2016) |
| :--- | :--- | :--- | :--- | :--- |
| **Region Proposal Mechanism** | Selective Search (CPU, slow) | Selective Search (CPU) | Region Proposal Network (RPN, GPU) | Single dense grid regression |
| **Feature Extraction** | Warps 2000 crops through CNN | Passes image once through CNN | Passes image once through CNN | Single end-to-end forward pass |
| **ROI Pooling** | None (pixel warping) | RoIPooling layer | RoIPooling / RoIAlign | Grid cell mapping |
| **Inference Speed** | ~40 - 50 seconds/image | ~2 seconds/image | ~0.2 seconds/image (5-7 FPS) | Real-time (30 - 60 FPS) |
| **End-to-End Trainable?** | No (Multi-stage) | Mostly (fixed proposals) | Yes (Fully end-to-end) | Yes (Fully end-to-end) |

### Key Takeaways from this Lab:
1. **Region proposals** dramatically reduce the search space from billions of sliding windows to ~300-2,000 promising regions.
2. **Transfer learning** with deep CNNs (e.g. ResNet-18) allows training high-accuracy object detectors even with small target datasets.
3. **Bounding box regression** using log-space scaling and normalized coordinate shifts significantly enhances spatial localization precision beyond coarse region proposal boundaries.
4. **Non-Maximum Suppression (NMS)** is an essential post-processing step that suppresses redundant multi-scale candidate proposals while preserving accurate true positive detections.
